# 00 · Ingesta en la nube: landing → Bronze

**Proyecto:** Minería aurífera y deforestación en el corredor La Pampa, Huepetuhe y Mazuko (Madre de Dios, Perú).

En este notebook todo ocurre dentro de Databricks:

1. **Landing:** los archivos oficiales (SERNANP, Geobosques-MINAM, GADM, OpenStreetMap) llegan al Volume tal como los publica cada institución.
2. **Ingesta cloud-to-cloud:** Sentinel-2 (AWS Open Data) y Hansen GFC (Google Cloud Storage) se leen directamente desde sus nubes de origen, solo los bloques del área de estudio.
3. **Bronze:** todo queda recortado y alineado a una grilla común UTM 19S de 20 m, en teselas de 512 px.

| Zona | Contenido | Granularidad |
|---|---|---|
| Landing | zip/shp/tif/xlsx originales | archivo |
| Bronze | teselas GeoTIFF alineadas + vectores recortados + metadatos | tesela |
| Silver | compuestos anuales sin nubes, índices espectrales, capas oficiales | píxel de 20 m |
| Gold | indicadores por hexágono H3 + covariables | hexágono (~0.1 km²) |

In [ ]:
%pip install -q rasterio pyproj h3 geopandas shapely esda libpysal spreg pykrige scikit-gstat mapclassify pystac-client openpyxl
dbutils.library.restartPython()

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Raíz del repositorio (Git folder en Databricks o clon local)
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from geocloud import config as C, dbx
spark = dbx.get_spark()
VOL = dbx.volume_path()
FIG = Path(VOL) / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Databricks:", dbx.is_databricks(), "| Volume:", VOL)

## 1. Unity Catalog y zona landing

In [ ]:
dbx.setup_catalog(spark)
LANDING, BRONZE = Path(VOL) / "landing", Path(VOL) / "bronze"
BRONZE.mkdir(parents=True, exist_ok=True)
grid = C.Grid()
print(f"Grilla: {grid.width} x {grid.height} píxeles de {grid.res} m = {grid.width*grid.height/1e6:.1f} M por banda, "
      f"{grid.n_tiles[0]} x {grid.n_tiles[1]} teselas de {grid.tile} px")

import pandas as pd
inv = pd.DataFrame([{"fuente": p.parent.name, "archivo": p.name, "MB": round(p.stat().st_size / 1e6, 1)}
                    for p in sorted(LANDING.rglob("*")) if p.is_file()])
dbx.show(inv)

## 2. Fuentes oficiales: landing → Bronze

- **SERNANP:** ANP de administración nacional definitivas y zonas de amortiguamiento.
- **Geobosques (MINAM):** pérdida anual 2001-2025 (separa pérdida antrópica y natural en 2025), cobertura bosque/no bosque 2025 y mapa de riesgo de deforestación.
- **GADM 4.1:** distritos. **OpenStreetMap (Geofabrik):** vías principales y ríos.

Los shapefiles se leen dentro del zip con GDAL (`/vsizip`) y con filtro espacial, sin descomprimir el país completo.

In [ ]:
from geocloud import ingest
resumen_ingesta = ingest.ingest_all(LANDING, BRONZE)
resumen_ingesta

## 3. Descarga satelital cloud-to-cloud

Sentinel-2 y Hansen están publicados como *Cloud Optimized GeoTIFF*: con lecturas HTTP por rangos
se transfieren solo los bloques del área de estudio (≈ 2 % de cada escena).

En Databricks Free Edition la salida a internet requiere verificar la cuenta con LinkedIn.
Si no está disponible, se usa el plan B: `scripts/download_data.py` en local y `databricks fs cp` al Volume.

In [ ]:
from geocloud import download
DESCARGAR = True          # False para reutilizar lo que ya está en el Volume

conectividad = download.has_internet()
print(conectividad)
if DESCARGAR and all(conectividad.values()):
    download.run_hansen(BRONZE)
    items = download.run_s2(BRONZE, years=C.YEARS, max_items=C.MAX_ITEMS_PER_YEAR, workers=8)
elif not (BRONZE / "s2").exists():
    print("Sin salida a internet y sin datos en bronze/s2: ejecutar el plan B (download_data.py + databricks fs cp).")
else:
    print("Usando las teselas Sentinel-2 y Hansen ya cargadas en el Volume.")

## 4. Inventario distribuido (sin abrir los archivos)

`binaryFile` lista los GeoTIFF como un DataFrame distribuido. Spark infiere las columnas `year` y `tile`
de la estructura `year=YYYY/tile=rRR_cCC` (particionamiento estilo Hive).

In [ ]:
from pyspark.sql import functions as F

s2_files = (spark.read.format("binaryFile").option("pathGlobFilter", "*.tif")
            .load(f"{VOL}/bronze/s2").drop("content")
            .withColumn("item_id", F.regexp_extract("path", r"([^/]+)\.tif$", 1)))
dbx.show(s2_files.groupBy("year").agg(F.countDistinct("item_id").alias("escenas"), F.count("*").alias("teselas"),
                                      F.round(F.sum("length") / 1e9, 2).alias("GB")).orderBy("year"))
tot = s2_files.agg(F.count("*").alias("n"), F.sum("length").alias("b")).first()
n_val = s2_files.select("year", "item_id").distinct().count() * grid.width * grid.height * 7
print(f"Archivos: {tot.n:,} | {tot.b/1e9:.2f} GB comprimidos | {n_val/1e9:.2f} mil millones de valores raster")

## 5. Tablas Bronze

In [ ]:
items = pd.read_csv(BRONZE / "s2_items.csv")
items = items[items["status"] == "descargado"]
dbx.save_table(spark.createDataFrame(items.astype({"year": int})), "bronze_s2_items")
dbx.save_table(s2_files, "bronze_s2_files")

import geopandas as gpd
for f in sorted((BRONZE / "vector").glob("*.geojson")):
    g = gpd.read_file(f)
    g["wkt"] = g.geometry.to_wkt()
    dbx.save_table(spark.createDataFrame(pd.DataFrame(g.drop(columns="geometry")).astype(str)), "bronze_vec_" + f.stem)
    print(f"bronze_vec_{f.stem:30s} {len(g):5d} geometrías")

In [ ]:
# Tablas oficiales de Geobosques: pérdida por distrito (para contrastar con nuestras estimaciones)
for t in sorted((BRONZE / "tablas").glob("*Distrito*.xlsx")):
    tab = pd.read_excel(t)
    mdd = tab[tab.astype(str).apply(lambda r: r.str.upper().str.contains("MADRE DE DIOS")).any(axis=1)]
    print(t.name, "| filas Madre de Dios:", len(mdd))
    dbx.show(mdd.head(20))

## 6. Control de calidad visual

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from matplotlib.colors import ListedColormap

sample = (spark.table(dbx.tbl("bronze_s2_files")).orderBy(F.desc("length"))
          .select("path").first().path.replace("dbfs:", "").replace("file:", ""))
tile = sample.split("tile=")[1].split("/")[0]
with rasterio.open(sample) as ds:
    arr = ds.read().astype(float)
rgb = np.clip(np.dstack([arr[2], arr[1], arr[0]]) / 10000, 0, None)
rgb = np.clip(rgb / np.nanpercentile(rgb[rgb > 0], 98), 0, 1)

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(rgb); ax[0].set_title("Sentinel-2 color verdadero (B4, B3, B2)")
ax[1].imshow(arr[6], cmap="tab20", vmin=0, vmax=11); ax[1].set_title("Máscara de escena SCL (nubes 8-10)")
gb = BRONZE / "geobosques" / f"tile={tile}" / "geobosques.tif"
if gb.exists():
    with rasterio.open(gb) as ds:
        per = ds.read(1).astype(float)
    per[(per == 0) | (per > 26)] = np.nan
    im = ax[2].imshow(per, cmap="plasma", vmin=1, vmax=26); plt.colorbar(im, ax=ax[2], fraction=0.046)
    ax[2].set_title("Geobosques: año de pérdida (1 = 2001 ... 25 = 2025)")
for a in ax: a.axis("off")
plt.suptitle(f"Tesela {tile}"); plt.tight_layout()
plt.savefig(FIG / "00_control_calidad.png", dpi=150); plt.show()